In [1]:
import os,sys
import pandas as pd
root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(root)
from utils import utils,models
from tqdm import tqdm

In [2]:
dataset_dir = '../dataset/newsimages_test_and_evaluation_26_v1.0'
images_dir = f'{dataset_dir}/news_images_evaluation'
dataset = pd.read_csv(f"{dataset_dir}/news_articles_test.csv",encoding='latin1')
dataset.set_index('article_id', inplace=True)
dataset['article_title'] = dataset['article_title'].apply(utils.normalize_text)

In [3]:
generator = models.OpenAIPipeline(model="gpt-5-mini")

In [5]:
results = []
checkpoint_path = "../dataset/vgqe_titles.json"

try:
    results = utils.load_json(checkpoint_path)
    start_idx = len(results)
    print(f"Resuming from {start_idx}")
except:
    results = []
    start_idx = 0

for i, row in enumerate(tqdm(dataset.itertuples(), total=len(dataset), desc="QE Generation")):
    
    if i < start_idx:
        continue

    query = row.article_title
    expanded = utils.clean_prompt(utils.generate_expansion_by_openai(generator, query))
    
    results.append({
        query:expanded
    })

    # Save checkpoint every 100 iterations
    if (i + 1) % 100 == 0:
        utils.save_json(results, checkpoint_path)

# Final save (important)
utils.save_json(results, checkpoint_path)

QE Generation: 100%|██████████| 800/800 [22:25<00:00,  1.68s/it]
